# 06 — Monte Carlo Tournament Simulation
## From Ability Estimates to Win Probabilities

**Purpose:** Convert posterior ability distributions into tournament outcome probabilities (P(win), P(top 5), P(make cut)) via Monte Carlo simulation.

**Process:**
1. Draw player abilities from posterior
2. Add course-fit adjustment
3. Simulate 4 rounds with Student-t noise + round autocorrelation
4. Apply cut rules (top 65 + ties)
5. Determine winner (playoff for ties)
6. Repeat 100,000+ times
7. P(win) = wins / simulations


In [ ]:
# --- Setup ---
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config.settings import Settings
from simulation.monte_carlo import MonteCarloSimulator, PlayerAbility
from simulation.tournament import TournamentConfig

cfg = Settings()


In [ ]:
# --- Create Synthetic Test Field ---
# In production, PlayerAbility comes from the Bayesian model posterior.
# Here we create a synthetic 30-player field to test the simulator.

np.random.seed(42)
n_players = 30

field = []
for i in range(n_players):
    # Skill: top players ~1.5 SG, average ~0, weak ~-0.5
    mu = np.random.normal(0.0, 0.7)
    field.append(PlayerAbility(
        player_id=1000 + i,
        mu_mean=mu,
        mu_std=abs(mu) * 0.2 + 0.1,   # More uncertainty for extreme estimates
        sigma=1.2 + np.random.uniform(-0.2, 0.2),  # Round noise ~1.2 strokes
        course_fit_mean=np.random.normal(0, 0.15),
        course_fit_std=0.1,
    ))

# Sort by skill for display
field.sort(key=lambda p: p.mu_mean, reverse=True)
print(f"Synthetic field: {n_players} players")
print(f"Top player:    μ={field[0].mu_mean:+.3f} SG")
print(f"Median player: μ={field[n_players//2].mu_mean:+.3f} SG")
print(f"Worst player:  μ={field[-1].mu_mean:+.3f} SG")


In [ ]:
# --- Run Simulation ---
sim = MonteCarloSimulator(cfg)

tourney = TournamentConfig(
    event_id=9999,
    event_name="Test Tournament",
    field_size=n_players,
    cut_rule="top_65_ties",  # Everyone makes cut with 30 players
)

# Run with 50K sims (faster for testing; use 100K+ for production)
result = sim.simulate_tournament(field, tourney, n_simulations=50_000)

# Display results
results_df = sim.results_to_dataframe(result)
print(f"\n=== SIMULATION RESULTS ({result.n_simulations:,} sims) ===")
print(f"Sum P(win) = {sum(result.win_probs.values()):.4f} (should be ~1.0)")
print(f"Convergence: {result.convergence_diagnostics}")
print(f"\nTop 15:")
print(results_df.head(15).to_string(index=False))


In [ ]:
# --- Win Probability Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: P(win) vs skill
skills = [p.mu_mean for p in field]
probs = [result.win_probs[p.player_id] for p in field]
axes[0].scatter(skills, probs, alpha=0.7, color="#2563eb", s=50)
axes[0].set_xlabel("True Skill (SG/round)")
axes[0].set_ylabel("P(win)")
axes[0].set_title("Win Probability vs Skill")

# Right: P(win) histogram
axes[1].bar(range(n_players), sorted(probs, reverse=True), color="#2563eb", alpha=0.7)
axes[1].set_xlabel("Player rank")
axes[1].set_ylabel("P(win)")
axes[1].set_title("Win Probability Distribution")

plt.tight_layout()
plt.show()


---
## ✅ Simulation Engine Working

**Key checks:**
- P(win) sums to ~1.0 ✓
- Higher-skilled players have higher P(win) ✓
- Convergence diagnostics show sufficient iterations ✓

**In production:** Replace synthetic field with real posterior estimates from Notebook 04.

**Next step:** → **07_betting_integration.ipynb**
